In [1]:
import sys
!{sys.executable} -m pip install scikit-learn pandas numpy matplotlib seaborn joblib 

In [2]:
import sys
!{sys.executable} -m pip install xgboost

In [3]:
# --- Core ---
import pandas as pd
import numpy as np

# --- Visualization (for outlier/distribution checks) ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing ---
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Train/test split + cross-validation ---
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV

# --- Models ---
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# --- Evaluation metrics ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Saving the final pipeline ---
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

# Display settings (optional, just makes wide dataframes readable)
pd.set_option('display.max_columns', None)

print("Setup complete — scikit-learn version:", __import__('sklearn').__version__)

Setup complete — scikit-learn version: 1.9.0


In [4]:
# we get that from task 1 notebook 
df = pd.read_csv('uber_cleaned.csv')

In [5]:
print(df.shape)

(482519, 25)


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 482519 entries, 0 to 482518
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Car Condition      482519 non-null  str    
 1   Weather            482519 non-null  str    
 2   Traffic Condition  482519 non-null  str    
 3   key                482519 non-null  str    
 4   fare_amount        482519 non-null  float64
 5   pickup_datetime    482519 non-null  str    
 6   pickup_longitude   482519 non-null  float64
 7   pickup_latitude    482519 non-null  float64
 8   dropoff_longitude  482519 non-null  float64
 9   dropoff_latitude   482519 non-null  float64
 10  passenger_count    482519 non-null  int64  
 11  hour               482519 non-null  int64  
 12  day                482519 non-null  int64  
 13  month              482519 non-null  int64  
 14  weekday            482519 non-null  int64  
 15  year               482519 non-null  int64  
 16  jfk_dist     

In [7]:
import sys
!{sys.executable} -m pip install statsmodels

In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Column selection (choose only what matters)

In [9]:
selected_cols = ['distance', 'jfk_dist', 'lga_dist', 'ewr_dist', 'nyc_dist', 'year']
X = df[selected_cols]
y = df['fare_amount']
print(X.head())

   distance   jfk_dist   lga_dist   ewr_dist   nyc_dist  year
0  1.030764  20.265840  14.342611  55.176046  27.572573  2009
1  8.450134  44.667679  23.130775  31.832358   8.755732  2010
2  1.389525  43.597686  19.865289  33.712082   9.847344  2011
3  2.799270  42.642965  21.063132  32.556289   7.703421  2012
4  1.999157  43.329953  15.219339  39.406828  15.600745  2010


## Chronological train/test split

In [10]:
df_sorted = df.sort_values('pickup_datetime')
X_sorted = df_sorted[selected_cols]
y_sorted = df_sorted['fare_amount']

split_idx = int(len(df_sorted) * 0.8)
X_train, X_test = X_sorted.iloc[:split_idx], X_sorted.iloc[split_idx:]
y_train, y_test = y_sorted.iloc[:split_idx], y_sorted.iloc[split_idx:]

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Train shape: (386015, 6) | Test shape: (96504, 6)


## Class for Outlier capping (will be used inside the Pipeline)

In [11]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_pct=0.01, upper_pct=0.99, exclude_cols=None):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct
        self.exclude_cols = exclude_cols or []

    def fit(self, X, y=None):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        cols_to_cap = [c for c in X_df.columns if c not in self.exclude_cols]
        self.cols_to_cap_ = cols_to_cap
        self.lower_bounds_ = X_df[cols_to_cap].quantile(self.lower_pct)
        self.upper_bounds_ = X_df[cols_to_cap].quantile(self.upper_pct)
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        outlier_mask = (
            X_df[self.cols_to_cap_].lt(self.lower_bounds_, axis=1) |
            X_df[self.cols_to_cap_].gt(self.upper_bounds_, axis=1)
        )
        num_outlier_values = outlier_mask.sum().sum()
        total_values = X_df[self.cols_to_cap_].shape[0] * X_df[self.cols_to_cap_].shape[1]
        pct_outlier_values = (num_outlier_values / total_values) * 100
        rows_with_outliers = outlier_mask.any(axis=1).sum()
        pct_rows = (rows_with_outliers / len(X_df)) * 100

        print("=" * 40)
        print("Outlier Report")
        print("=" * 40)
        print(f"Outlier values           : {num_outlier_values}")
        print(f"Percentage of values     : {pct_outlier_values:.2f}%")
        print(f"Rows with outliers       : {rows_with_outliers}")
        print(f"Percentage of rows       : {pct_rows:.2f}%")
        print("=" * 40)

        X_df[self.cols_to_cap_] = X_df[self.cols_to_cap_].clip(
            lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1
        )
        return X_df

In [12]:
# Standalone check — see the outlier report before it's folded into the full pipeline
capper = OutlierCapper(lower_pct=0.01, upper_pct=0.99)
capper.fit(X_train)
X_train_capped = capper.transform(X_train)

# Compare summary stats before vs after capping
print("\nBEFORE capping:")
print(X_train.describe())

print("\nAFTER capping:")
print(X_train_capped.describe())

Outlier Report
Outlier values           : 38610
Percentage of values     : 1.67%
Rows with outliers       : 25163
Percentage of rows       : 6.52%

BEFORE capping:
            distance       jfk_dist       lga_dist       ewr_dist  \
count  386015.000000  386015.000000  386015.000000  386015.000000   
mean        3.348370      41.907356      19.557519      35.603337   
std         3.784422       4.780417       4.559197       5.630822   
min         0.000084       1.017646       0.382119       1.460945   
25%         1.279231      41.330249      17.068341      32.109139   
50%         2.176437      42.485405      19.497293      34.636313   
75%         3.941741      43.672039      22.006575      37.978797   
max       110.833077     287.225253     253.242545     261.570413   

            nyc_dist           year  
count  386015.000000  386015.000000  
mean       11.095442    2011.090323  
std         6.095290       1.474553  
min         0.143207    2009.000000  
25%         7.055466    

In [13]:
# Count exactly how many values differ between original and capped
changed = (X_train != X_train_capped).sum()
print("Values changed per column:\n", changed)

Values changed per column:
 distance    7722
jfk_dist    7722
lga_dist    7722
ewr_dist    7722
nyc_dist    7722
year           0
dtype: int64


## VIF-based multicollinearity filter

In [14]:
import statsmodels.api as sm
"""Iteratively drops the column with the highest VIF (above threshold),
    learned from TRAINING data only. Applies the same kept-column list to
    any future data (test set, new predictions)."""

class VIFDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=7.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=selected_cols)
        cols = list(X_df.columns)

        while len(cols) > 1:
            X_with_const = sm.add_constant(X_df[cols])  # add intercept term
            vif_values = [variance_inflation_factor(X_with_const.values, i)
                          for i in range(1, X_with_const.shape[1])]  # skip index 0 = the constant itself
            max_vif = max(vif_values)
            if max_vif > self.threshold:
                drop_col = cols[vif_values.index(max_vif)]
                print(f"Dropping '{drop_col}' due to VIF = {max_vif:.2f}")
                cols.remove(drop_col)
            else:
                break

        self.cols_to_keep_ = cols
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=selected_cols)
        return X_df[self.cols_to_keep_]

In [15]:
# Checking the VIF values directly before they're auto-handled inside the pipeline
checker = VIFDropper(threshold=7.0)
checker.fit(X_train)
print("Columns kept after VIF filtering:", checker.cols_to_keep_)

Dropping 'ewr_dist' due to VIF = 14.86
Columns kept after VIF filtering: ['distance', 'jfk_dist', 'lga_dist', 'nyc_dist', 'year']


In [16]:
from sklearn.linear_model import LinearRegression

baseline_pipeline = Pipeline(steps=[
    ('outlier_capping', OutlierCapper(lower_pct=0.01, upper_pct=0.99)),
    ('vif_filter', VIFDropper(threshold=10.0)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

print("Baseline MAE:", mean_absolute_error(y_test, y_pred_baseline))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_baseline)))
print("Baseline R²:", r2_score(y_test, y_pred_baseline))

Outlier Report
Outlier values           : 38610
Percentage of values     : 1.67%
Rows with outliers       : 25163
Percentage of rows       : 6.52%
Dropping 'ewr_dist' due to VIF = 33.50
Outlier Report
Outlier values           : 44025
Percentage of values     : 7.60%
Rows with outliers       : 38284
Percentage of rows       : 39.67%
Baseline MAE: 2.694782343688186
Baseline RMSE: 5.10050725715256
Baseline R²: 0.7879960391304457


## Building the full pipeline

In [17]:
full_pipeline = Pipeline(steps=[
    ('outlier_capping', OutlierCapper(lower_pct=0.01, upper_pct=0.99, exclude_cols=['year'])),
    ('vif_filter', VIFDropper(threshold=10.0)),
    ('scaler', StandardScaler()),
    ('model', HistGradientBoostingRegressor(random_state=42))
])


In [18]:
param_distributions = {
    'model__max_iter': [100, 200, 300],
    'model__max_depth': [3, 5, 7, None],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__l2_regularization': [0, 0.1, 1.0]
}

tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [19]:
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Outlier Report
Outlier values           : 38610
Percentage of values     : 2.00%
Rows with outliers       : 25163
Percentage of rows       : 6.52%
Dropping 'ewr_dist' due to VIF = 33.50
Best parameters: {'model__max_iter': 200, 'model__max_depth': 7, 'model__learning_rate': 0.2, 'model__l2_regularization': 0.1}


In [20]:
best_pipeline = search.best_estimator_
y_pred = best_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

Outlier Report
Outlier values           : 10060
Percentage of values     : 2.08%
Rows with outliers       : 6742
Percentage of rows       : 6.99%
MAE:  2.1122
RMSE: 4.0049
R²:   0.8693


In [21]:
joblib.dump(best_pipeline, 'simple_uber_pipeline.joblib')
print("Pipeline saved.")

Pipeline saved.


In [22]:

loaded_pipeline = joblib.load('simple_uber_pipeline.joblib')
print(loaded_pipeline)

Pipeline(steps=[('outlier_capping', OutlierCapper(exclude_cols=['year'])),
                ('vif_filter', VIFDropper(threshold=10.0)),
                ('scaler', StandardScaler()),
                ('model',
                 HistGradientBoostingRegressor(l2_regularization=0.1,
                                               learning_rate=0.2, max_depth=7,
                                               max_iter=200,
                                               random_state=42))])


In [24]:
# Quick sanity test directly in a notebook cell, using your loaded pipeline
import pandas as pd

test_cases = pd.DataFrame([
    {'distance': 2.0, 'jfk_dist': 15, 'lga_dist': 10, 'ewr_dist': 20, 'nyc_dist': 5, 'year': 2014},
    {'distance': 25.0, 'jfk_dist': 0.12, 'lga_dist': 0.11, 'ewr_dist': 0.16, 'nyc_dist': 0.17, 'year': 2014},
    {'distance': 0.5, 'jfk_dist': 15, 'lga_dist': 10, 'ewr_dist': 20, 'nyc_dist': 5, 'year': 2014},
])

predictions = loaded_pipeline.predict(test_cases)
print(predictions)

Outlier Report
Outlier values           : 9
Percentage of values     : 60.00%
Rows with outliers       : 3
Percentage of rows       : 100.00%
[ 7.93416504 44.75152363  6.59094761]
